# Build a read-only library tool adapter

A library assistant offers two read-only tools, `search_catalogue` and `check_availability`. The model has just answered a request with several **parallel tool calls**, in the OpenAI-compatible chat-completions format from C6 lesson 1. Your job is the application side of the round trip: read the calls, run the allowed ones safely, and build request 2 with one `tool` message per call.

Everything is a recorded fixture. There are no network calls and no API keys. One call in each fixture fails in a different way: a timeout, invalid arguments or an unknown tool. Your code has to turn each failure into **error content** the model can read, not a crash and not a silence.

Fill in each `TODO` cell, then run the check cell after it. Before submitting, restart the kernel and run all cells. Standard library only.

## Setup: tools, handlers and fixtures

Run this cell as given. It defines:

- `RECORDS`: synthetic catalogue rows. Some are private, and all have an internal `staff_note`.
- Two validators that return clean arguments or `None` (you wrote one like these in the lesson 1 exercise).
- Two handlers. `check_availability` raises `TimeoutError` for the `harbour` branch, standing in for a slow service.
- `REGISTRY`, mapping each tool name to a `(validator, handler)` pair.
- Two fixture sets, `FIXTURE_A` and `FIXTURE_B`. Each has a `request` (request 1) and a `response` (response 1).

In [ ]:
import json

RECORDS = [
    {"id": "m1", "title": "Maps of the Coast", "public": True, "staff_note": "spine taped"},
    {"id": "m2", "title": "Secret Maps", "public": False, "staff_note": "restricted"},
    {"id": "m3", "title": "Road Maps of Wales", "public": True, "staff_note": ""},
    {"id": "c1", "title": "Café Society: Paris Between the Wars", "public": True, "staff_note": ""},
    {"id": "c2", "title": "Cafés of Lisbon and Porto", "public": True, "staff_note": ""},
    {"id": "c3", "title": "Café Culture in Vienna, 1890 to 1938", "public": True, "staff_note": ""},
]
STOCK = {("9780141439518", "central"): 2, ("9780141439518", "north"): 0}
BRANCHES = {"central", "north", "harbour"}

def search_arguments(raw):
    try:
        args = json.loads(raw)
    except json.JSONDecodeError:
        return None
    if not isinstance(args, dict) or set(args) != {"query", "limit"}:
        return None
    q, limit = args["query"], args["limit"]
    if not isinstance(q, str) or not q.strip() or type(limit) is not int or not 1 <= limit <= 5:
        return None
    return {"query": q.strip(), "limit": limit}

def availability_arguments(raw):
    try:
        args = json.loads(raw)
    except json.JSONDecodeError:
        return None
    if not isinstance(args, dict) or set(args) != {"isbn", "branch"}:
        return None
    isbn, branch = args["isbn"], args["branch"]
    if not (isinstance(isbn, str) and len(isbn) == 13 and isbn.isascii() and isbn.isdigit()):
        return None
    if not isinstance(branch, str) or branch not in BRANCHES:
        return None
    return args

def search_catalogue(query, limit):
    q = query.casefold()
    hits = [{"id": r["id"], "title": r["title"]}
            for r in RECORDS if r["public"] and q in r["title"].casefold()]
    return hits[:limit]

def check_availability(isbn, branch):
    if branch == "harbour":
        raise TimeoutError("harbour branch service did not answer in 2 s")
    return {"isbn": isbn, "branch": branch, "copies": STOCK.get((isbn, branch), 0)}

REGISTRY = {
    "search_catalogue": (search_arguments, search_catalogue),
    "check_availability": (availability_arguments, check_availability),
}

def _call(call_id, name, args_text):
    return {"id": call_id, "type": "function",
            "function": {"name": name, "arguments": args_text}}

def _response(calls, finish_reason="tool_calls"):
    return {"choices": [{"index": 0, "finish_reason": finish_reason, "message": {
        "role": "assistant", "content": None, "tool_calls": calls}}]}

SYSTEM = {"role": "system", "content": "You help library members. Use tools for catalogue questions."}

FIXTURE_A = {
    "request": {"model": "example-model", "messages": [
        SYSTEM, {"role": "user", "content": "Find maps books and check 9780141439518 at harbour."}]},
    "response": _response([
        _call("call_A1", "search_catalogue", '{"query": "maps", "limit": 3}'),
        _call("call_A2", "check_availability", '{"isbn": "9780141439518", "branch": "harbour"}'),
        _call("call_A3", "search_catalogue", '{"query": "maps", "limit": 9}'),
    ]),
}

FIXTURE_B = {
    "request": {"model": "example-model", "messages": [
        SYSTEM, {"role": "user", "content": "Is 9780141439518 in at central? Renew my loan. Any cafe books?"}]},
    "response": _response([
        _call("call_9", "check_availability", '{"isbn": "9780141439518", "branch": "central"}'),
        _call("call_2", "renew_loan", '{"loan_id": "L-77"}'),
        _call("call_5", "search_catalogue", '{"query": "café", "limit": 5}'),
    ]),
}
print("setup OK")

## 1. Read the proposed calls

Write `read_tool_calls(response)`. If `choices[0].finish_reason` is `"tool_calls"`, return `choices[0].message.tool_calls` as a list. For any other finish reason, return an empty list, because the model answered in text and there is nothing to run.

In [ ]:
def read_tool_calls(response):
    # TODO
    raise NotImplementedError

In [ ]:
assert [c["id"] for c in read_tool_calls(FIXTURE_A["response"])] == ["call_A1", "call_A2", "call_A3"]
assert [c["id"] for c in read_tool_calls(FIXTURE_B["response"])] == ["call_9", "call_2", "call_5"]
text_reply = {"choices": [{"index": 0, "finish_reason": "stop",
                           "message": {"role": "assistant", "content": "Here you go."}}]}
assert read_tool_calls(text_reply) == [], "A stop response has no calls to run"
print("OK")

## 2. Run one call and classify the outcome

Write `run_call(call, registry)`. It returns an **outcome** dict:

| Situation | Outcome |
|---|---|
| `function.name` is not in `registry` | `{"status": "error", "error": "unknown_tool"}` |
| The validator returns `None` | `{"status": "error", "error": "invalid_arguments"}` |
| The handler raises `TimeoutError` | `{"status": "error", "error": "timeout"}` |
| The handler returns a value | `{"status": "ok", "result": value}` |

Look the name up in the registry, never with `globals()`, `getattr` or `eval`. Call the handler with the validator's cleaned arguments as keyword arguments. Catch **only** `TimeoutError`: any other exception is a bug and must propagate.

In [ ]:
def run_call(call, registry):
    # TODO
    raise NotImplementedError

In [ ]:
a1, a2, a3 = read_tool_calls(FIXTURE_A["response"])
assert run_call(a1, REGISTRY) == {"status": "ok", "result": [
    {"id": "m1", "title": "Maps of the Coast"}, {"id": "m3", "title": "Road Maps of Wales"}]}
assert run_call(a2, REGISTRY) == {"status": "error", "error": "timeout"}
assert run_call(a3, REGISTRY) == {"status": "error", "error": "invalid_arguments"}
b9, b2, b5 = read_tool_calls(FIXTURE_B["response"])
assert run_call(b9, REGISTRY) == {"status": "ok", "result": {"isbn": "9780141439518", "branch": "central", "copies": 2}}
assert run_call(b2, REGISTRY) == {"status": "error", "error": "unknown_tool"}
assert run_call(_call("x", "__import__", '{"name": "os"}'), REGISTRY) == {"status": "error", "error": "unknown_tool"}

def broken(query, limit):
    return {}["missing key"]
try:
    run_call(a1, {"search_catalogue": (search_arguments, broken)})
except KeyError:
    pass
else:
    raise AssertionError("Catch only TimeoutError: a KeyError is a bug and must propagate")
print("OK")

## 3. Serialise an outcome under a byte cap

Write `tool_content(outcome, max_bytes)`, which returns the string for a tool message's `content`.

- Serialise with `json.dumps(outcome, ensure_ascii=False)` and measure **UTF-8 bytes**, not characters.
- If it fits in `max_bytes`, return it.
- If it doesn't fit and `outcome["result"]` is a list, drop items from the **end** of the list until it fits. Return the outcome with the shorter list and an added `"truncated": true`. Measure each candidate *with* the `truncated` key included.
- If it still doesn't fit (even with an empty list), or the result is not a list, return the serialised `{"status": "error", "error": "result_too_large"}`.

Never slice the JSON string itself: the model would receive invalid JSON with no sign that anything was lost.

In [ ]:
def tool_content(outcome, max_bytes):
    # TODO
    raise NotImplementedError

In [ ]:
small = {"status": "error", "error": "timeout"}
assert json.loads(tool_content(small, 100)) == small
cafes = run_call(read_tool_calls(FIXTURE_B["response"])[2], REGISTRY)
full = json.dumps(cafes, ensure_ascii=False)
assert len(full.encode("utf-8")) > len(full), "Titles contain é, so bytes exceed characters"
assert json.loads(tool_content(cafes, 500)) == cafes, "A result under the cap is unchanged"
tight = len(full.encode("utf-8")) - 1   # still more than len(full) characters
assert len(tool_content(cafes, tight).encode("utf-8")) <= tight, "Measure UTF-8 bytes, not characters"
for cap, kept in ((180, 2), (120, 1), (60, 0)):
    text = tool_content(cafes, cap)
    assert len(text.encode("utf-8")) <= cap, f"Content must fit in {cap} bytes"
    got = json.loads(text)
    assert got["truncated"] is True and got["result"] == cafes["result"][:kept], f"cap {cap}: keep the first {kept} items"
too_big = {"status": "ok", "result": {"isbn": "9780141439518", "branch": "central", "copies": 2}}
assert json.loads(tool_content(too_big, 40)) == {"status": "error", "error": "result_too_large"}
assert json.loads(tool_content(cafes, 10)) == {"status": "error", "error": "result_too_large"}
print("OK")

## 4. Build the tool messages

Write `tool_messages(response, registry, max_bytes)`. Return one message per proposed call, in the order the calls appear:

```python
{"role": "tool", "tool_call_id": <the call's id>, "content": <tool_content(run_call(...), max_bytes)>}
```

Every call ID must be answered, including failed calls. A response with no calls gives an empty list.

In [ ]:
def tool_messages(response, registry, max_bytes):
    # TODO
    raise NotImplementedError

In [ ]:
msgs_a = tool_messages(FIXTURE_A["response"], REGISTRY, 400)
assert [m["role"] for m in msgs_a] == ["tool"] * 3
assert [m["tool_call_id"] for m in msgs_a] == ["call_A1", "call_A2", "call_A3"]
assert [json.loads(m["content"]) for m in msgs_a] == [
    {"status": "ok", "result": [{"id": "m1", "title": "Maps of the Coast"}, {"id": "m3", "title": "Road Maps of Wales"}]},
    {"status": "error", "error": "timeout"},
    {"status": "error", "error": "invalid_arguments"},
]
assert all("staff_note" not in m["content"] and "Secret" not in m["content"] for m in msgs_a)

msgs_b = tool_messages(FIXTURE_B["response"], REGISTRY, 150)
assert [m["tool_call_id"] for m in msgs_b] == ["call_9", "call_2", "call_5"], "Keep the model's call order, not sorted IDs"
assert json.loads(msgs_b[0]["content"]) == {"status": "ok", "result": {"isbn": "9780141439518", "branch": "central", "copies": 2}}
assert json.loads(msgs_b[1]["content"]) == {"status": "error", "error": "unknown_tool"}
last = json.loads(msgs_b[2]["content"])
assert last["truncated"] is True and [r["id"] for r in last["result"]] == ["c1"]
assert all(len(m["content"].encode("utf-8")) <= 150 for m in msgs_b)
assert tool_messages({"choices": [{"index": 0, "finish_reason": "stop", "message": {"role": "assistant", "content": "Hi"}}]}, REGISTRY, 100) == []
print("OK")

## 5. Build request 2

Write `next_request(request, response, registry, max_bytes)`. Return a **new** request dict with the same keys as `request`, whose `messages` are:

1. the original messages,
2. the assistant message from `response`, unchanged,
3. the tool messages from task 4.

Do not modify `request` itself: a retry may need to resend it.

In [ ]:
def next_request(request, response, registry, max_bytes):
    # TODO
    raise NotImplementedError

In [ ]:
for name, fx, n_calls in (("A", FIXTURE_A, 3), ("B", FIXTURE_B, 3)):
    before = json.dumps(fx["request"])
    req2 = next_request(fx["request"], fx["response"], REGISTRY, 300)
    assert json.dumps(fx["request"]) == before, f"Fixture {name}: do not modify request 1"
    assert req2["model"] == fx["request"]["model"]
    roles = [m["role"] for m in req2["messages"]]
    assert roles == ["system", "user", "assistant"] + ["tool"] * n_calls, f"Fixture {name}: got roles {roles}"
    assert req2["messages"][2] == fx["response"]["choices"][0]["message"], "Send the assistant message back unchanged"
    issued = [c["id"] for c in req2["messages"][2]["tool_calls"]]
    answered = [m["tool_call_id"] for m in req2["messages"][3:]]
    assert issued == answered, f"Fixture {name}: every issued call ID needs exactly one tool message"
print("OK")

## 6. Interpretation

Answer in a few sentences each.

1. In fixture A, what do you expect the model to say to the user in its reply to request 2, given one success, one timeout and one invalid-argument error? Which of these errors could the model fix by itself on another round?
2. Why did `renew_loan` get `unknown_tool` rather than being forwarded to the library's loan system? What would have to change before a tool with side effects like that could be offered, and who should confirm it?
3. Which failures should reach the user, which should go only to your logs, and which to both? Consider the timeout, the `KeyError` bug from task 2 and the truncated café result.

**Your answer**